In [52]:
import spacy
import scispacy
from scispacy.linking import EntityLinker
from scispacy.abbreviation import AbbreviationDetector
import csv
from textblob import TextBlob
from textblob.np_extractors import ConllExtractor


In [53]:
nlp = spacy.load("en_core_sci_sm")
nlp.add_pipe("scispacy_linker", config={"linker_name": "umls"})
nlp.add_pipe("abbreviation_detector")

c:\Users\amara\Documents\personal-health-passport\mhp-env\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\amara\Documents\personal-health-passport\mhp-env\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.1.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [54]:

SEMANTIC_TYPES = {}

with open("semantic_types.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        SEMANTIC_TYPES[row["code"]] = row



In [232]:
text = """
The patient has lupus nephritis and was prescribed rituximab and MMF.
Proteinuria has improved following treatment.
"""

text2 = """
History:
The patient has lupus nephritis and hypertension. She was started on mycophenolate mofetil (MMF) and prednisone six months ago.

Assessment:
Proteinuria has significantly improved since treatment began. Renal function remains stable. The patient's fatigue has decreased, but the joint pain has worsened over the past two weeks. Blood pressure is well controlled. Serum creatinine has increased slightly compared with the previous visit. The rash has completely resolved. The patient denies fever, chest pain, and shortness of breath. Mild nausea developed after increasing the MMF dose but has since subsided.

Plan:
Continue MMF and prednisone. Reduce the steroid dose gradually. Repeat renal function tests in four weeks.
"""

doc = nlp(text)
doc2 = nlp(text2)

c:\Users\amara\Documents\personal-health-passport\mhp-env\Lib\site-packages\scispacy\abbreviation.py:248: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  global_matches = self.global_matcher(doc)


In [104]:
def get_semantic_class(code):
    info = SEMANTIC_TYPES[code]
    return info["category"]

In [140]:
def get_entity_info(doc):
    entities = {}
    linker = nlp.get_pipe("scispacy_linker")

    abbreviation_map = {
        str(abbr): str(abbr._.long_form)
        for abbr in doc._.abbreviations
    }

    print("Abbreviation Map:", abbreviation_map)
    
    for entity in doc.ents:
        entity_text = entity.text

        normalised_text = abbreviation_map.get(entity_text, entity_text)

        normalised_doc = nlp(normalised_text)

        if normalised_doc.ents:
            normalised_entity = normalised_doc.ents[0]

            if normalised_entity._.kb_ents:
                cui, score = normalised_entity._.kb_ents[0]
                concept = linker.kb.cui_to_entity[cui]

                # print("Original:", entity.text)
                # print("Normalised:", normalised_text)
                # print("Canonical:", concept.canonical_name)
                # print("Semantic types:", concept.types)
                # print("Score:", score)
                # print()

                for code in concept.types:
                    if get_semantic_class(code) not in entities:
                        entities[get_semantic_class(code)] = []
                    
                    entities[get_semantic_class(code)].append({
                        "entity": entity_text,
                        "normalised": normalised_text,
                        "canonical": concept.canonical_name,
                        "score": score
                    })

    return entities

In [ ]:
def check_negation(token):
    negation_words = {
        "deny",
        "no",
        "without",
        "negative",
        "absence"
    }

    if token.dep_ == "neg" or token.lemma_.lower() in negation_words:
        for child in token.children:
            if child.dep_ == "dobj":
                return {child.text,token.lemma_.lower()}
            
    return False

In [239]:
def extract_clinical_relationships(doc):
    improvement_words = {
        "improve",
        "resolve",
        "decrease",
        "reduce",
        "subside",
        "recover",
        "remit"
    }

    decline_words = {
        "worsen",
        "deteriorate",
        "decline",
        "progress",
        "exacerbate",
        "increase",
        "rise",
        "relapse",
        "develop"
    }

    stable_words = {
        "stable",
        "remain",
        "unchanged",
        "control"
    }

    negation_words = {
        "deny",
        "no",
        "without",
        "negative",
        "absence"
    }

    relationships = []

    for token in doc:

        lemma = token.lemma_.lower()
        # Find clinical status verbs
        if lemma in improvement_words | decline_words | stable_words | negation_words:

            status = None

            if lemma in improvement_words:
                status = "improved"

            elif lemma in decline_words:
                status = "declined"

            elif lemma in stable_words:
                status = "stable"

            elif lemma in negation_words:
                status = "negated"

            # Find the entity connected to the verb
            for child in token.children:
                                    
                if child.dep_ == "dobj" and status == "negated":
                    
                    relationships.append({
                        "entity": child.text,
                        "trigger": token.text,
                        "status": status
                    })

                    for grandchild in child.children:
                        if grandchild.dep_ == "conj":
                            relationships.append({
                                "entity": grandchild.text,
                                "trigger": token.text,
                                "status": status
                            }) 

                # Active voice
                elif child.dep_ == "nsubj" and status != "negated":
                    phrase = child.text
                    for grandchild in child.children:
                        if not grandchild.is_space and grandchild.dep_ in ("amod", "compound"):
                            phrase = grandchild.text + " " + child.text
                            break

                    relationships.append({
                        "entity": phrase,
                        "trigger": token.text,
                        "status": status
                    })


                # Passive voice
                elif child.dep_ == "nsubjpass":
                    phrase = child.text
                    for grandchild in child.children:
                        if not grandchild.is_space and grandchild.dep_ in ("amod", "compound"):
                            phrase = grandchild.text + " " + child.text
                            break

                    relationships.append({
                        "entity": phrase,
                        "trigger": token.text,
                        "status": status
                    })


                


    return relationships
        
for relationship in extract_clinical_relationships(doc2):
    print(relationship)


{'entity': 'Proteinuria', 'trigger': 'improved', 'status': 'improved'}
{'entity': 'Renal function', 'trigger': 'remains', 'status': 'stable'}
{'entity': 'fatigue', 'trigger': 'decreased', 'status': 'improved'}
{'entity': 'joint pain', 'trigger': 'worsened', 'status': 'declined'}
{'entity': 'Blood pressure', 'trigger': 'controlled', 'status': 'stable'}
{'entity': 'Serum creatinine', 'trigger': 'increased', 'status': 'declined'}
{'entity': 'rash', 'trigger': 'resolved', 'status': 'improved'}
{'entity': 'fever', 'trigger': 'denies', 'status': 'negated'}
{'entity': 'Mild nausea', 'trigger': 'developed', 'status': 'declined'}


In [ ]:
def organise_ents(doc ,n_ents):
    result = {}

    # positive = nlp("improved")
    # neutral = nlp("unchanged")
    # negative = nlp("worsened")

    for semantic_class, entity_list in n_ents.items():
        print(f"{semantic_class}:")
    
        for item in entity_list:
            print(item["entity"])
        print("-" * 30)

    # for token in doc:

    #     similarities = [positive.similarity(token), neutral.similarity(token), negative.similarity(token)]
    #     decider = max(similarities);
    #     for i, sim in enumerate(similarities):
    #         if sim == decider and sim > 0.5:
    #             if i == 0:
    #                 for child in token.children:
    #                     if child.dep_ == "nsubj" and child.text in n_ents:
    #                         print("Improvement verb:", token.text)
    #                         print("Affected entity:", child.text)
    #                         result[child.text] = {
    #                             "normalised": n_ents[child.text]["normalised"],
    #                             "Status": "improved"}
    #             elif i == 1:      
    #                 for child in token.children:
    #                     if child.dep_ == "nsubj" and child.text in n_ents:
    #                         print("Neutral verb:", token.text)
    #                         print("Affected entity:", child.text)
    #                         result[child.text] = {
    #                             "normalised": n_ents[child.text]["normalised"],
    #                             "Status": "unchanged"}
    #             elif i == 2:              
    #                 for child in token.children:
    #                     if child.dep_ == "nsubj" and child.text in n_ents:
    #                         print("Negative verb:", token.text)
    #                         print("Affected entity:", child.text)
    #                         result[child.text] = {
    #                             "normalised": n_ents[child.text]["normalised"],
    #                             "Status": "worsened"}

           
            
            

In [ ]:
organise_ents(doc2, get_entity_info(doc2))

[{'entity': 'Proteinuria', 'trigger': 'improved', 'status': 'improved'},
 {'entity': 'function', 'trigger': 'remains', 'status': 'stable'},
 {'entity': 'fatigue', 'trigger': 'decreased', 'status': 'improved'},
 {'entity': 'pain', 'trigger': 'worsened', 'status': 'declined'},
 {'entity': 'creatinine', 'trigger': 'increased', 'status': 'declined'},
 {'entity': 'rash', 'trigger': 'resolved', 'status': 'improved'}]